# LexSelect API — Upload & Process

This notebook walks through the document upload flow:
1. Upload a document (single multipart request)
2. Poll processing status until done
3. Fetch page metadata and the parsed structure

**Prerequisites:** Set `LEXSELECT_API_KEY` in a `.env` file or environment variable.

In [ ]:
import os
import time
import httpx
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ["LEXSELECT_API_KEY"]
API_URL = os.environ.get("LEXSELECT_API_URL", "https://api.lexselect.io/api")

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "X-API-Version": "2026-03-06",
}

client = httpx.Client(base_url=API_URL, headers=headers, timeout=30)

# Map file extensions to the MIME type used for the uploaded file part. The
# server also infers the type from the file name when this is omitted.
CONTENT_TYPES = {
    "pdf": "application/pdf",
    "doc": "application/msword",
    "docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
    "rtf": "application/rtf",
    "html": "text/html",
    "msg": "application/vnd.ms-outlook",
    "eml": "message/rfc822",
    "odt": "application/vnd.oasis.opendocument.text",
}


def content_type_for(name: str) -> str:
    ext = name.rsplit(".", 1)[-1].lower() if "." in name else ""
    return CONTENT_TYPES.get(ext, "application/octet-stream")


print(f"Connected to {API_URL}")

## Health Check

Verify the API is reachable.

In [ ]:
resp = client.get("/health")
print(resp.json())

## Upload Document

Upload the file in a single multipart request (`POST /documents/upload`) — the server stores the bytes, verifies the hash, and starts processing.

In [ ]:
# Point this at your own document — the repo does not ship a sample file.
FILE_PATH = "../../sample.pdf"  # <-- change this to your file

file_name = os.path.basename(FILE_PATH)
file_size = os.path.getsize(FILE_PATH)

with open(FILE_PATH, "rb") as f:
    file_data = f.read()

# Single-request multipart upload (POST /documents/upload): the server stores
# the bytes, verifies the hash, and triggers processing. A standalone request is
# used so httpx sets the multipart Content-Type (the shared client defaults to
# application/json for the JSON endpoints).
resp = httpx.post(
    f"{API_URL}/documents/upload",
    headers={"Authorization": f"Bearer {API_KEY}", "X-API-Version": "2026-03-06"},
    data={"name": file_name, "size": str(file_size)},
    files={"file": (file_name, file_data, content_type_for(file_name))},
    timeout=60,
)
resp.raise_for_status()
doc = resp.json()

real_doc_id = doc["id"]
print(f"Uploaded: {real_doc_id}")
print(f"Name: {doc['name']}")
print(f"Type: {doc['type']}")

## Poll Processing Status

Wait for the document to finish processing.

In [ ]:
for i in range(120):
    time.sleep(3)
    resp = client.get(f"/documents/{real_doc_id}/processing/latest")

    if resp.status_code == 404:
        print(".", end="", flush=True)
        continue

    proc = resp.json()
    status = proc["status"]

    if status == "completed":
        print(f"\nProcessing complete!")
        print(f"  Pages: {proc['page_count']}")
        print(f"  Engine: v{proc['engine_version']}")
        print(f"  Preview: {proc['preview_available']}")
        break
    elif status == "failed":
        print(f"\nProcessing failed: {proc.get('error_message')}")
        break
    else:
        progress = proc.get('processing_progress', 0)
        print(f"  {status}: {progress:.0%}", end="\r", flush=True)
else:
    print("Timed out waiting for processing")

## Get Page Metadata

Fetch metadata for all pages of the processed document.

In [ ]:
resp = client.get(f"/documents/{real_doc_id}/processing/latest/pages")
resp.raise_for_status()
pages_data = resp.json()

print(f"Total pages: {pages_data['total']}\n")
for page in pages_data["pages"]:
    print(f"  Page {page['page_index']}: {page['width']}x{page['height']} ({page.get('page_type', 'unknown')})")

In [ ]:
# Parsed structure — the structured projection of the processed document
# (tree, key-value pairs, text, tables, blocks). This is the product value.
resp = client.get(f"/documents/{real_doc_id}/parse")
resp.raise_for_status()
parsed = resp.json()

print(f"Document type:  {parsed['document_type']}")
print(f"Status:         {parsed['status']}")
print(f"Pages parsed:   {parsed['page_count']} / {parsed['total_pages']}")

# Render to Markdown (experimental). Markdown may contain inline HTML from the
# source document — sanitize before rendering to HTML in a browser.
md = client.get(f"/documents/{real_doc_id}/render", params={"format": "markdown"})
md.raise_for_status()
print(f"\nMarkdown preview ({len(md.text)} chars):\n{md.text[:300]}")

## List Documents

See all your documents.

In [ ]:
resp = client.get("/documents", params={"limit": 10, "sort_by": "created_at", "sort_direction": "desc"})
resp.raise_for_status()
data = resp.json()

# Note: `total` is the count in THIS page, not the grand total across all pages.
# Page through results with `next_cursor` until it is null.
print(f"Documents (this page: {len(data['items'])}):")
for doc in data["items"]:
    print(f"  {doc['id']}  {doc['name']}  ({doc['type']})")